(ch:trainingModels)=
# 선형 회귀 모델 훈련

:::{note} 감사의 글

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 이미지 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.
:::

:::{seealso} 코드 실행
[(코드 워크아웃) 모델 훈련](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-training_models.ipynb)를 병행하여 읽을 것을 권장한다.
:::

선형 회귀 모델의 구조와 훈련 원리를 출발점으로 삼아 경사하강법의 작동 방식을 살펴본다.
이어서 다항 회귀를 통한 비선형 데이터 학습, 과소/과대 적합 판단, 모델 규제와 조기 종료 등 모델 훈련의 핵심 개념을 다룬다.
마지막으로 분류 문제에 활용되는 로지스틱 회귀와 소프트맥스 회귀를 소개한다.

## 선형 회귀 

선형 회귀 모델을 이용하여 머신러닝 모델의 기능과
훈련 과정의 이해에 중요한 기초 개념을 살펴 본다.
선형 회귀 모델을 예제로 사용하는 이유는 크게 두 가지다.

첫째, 선형 회귀 모델의 훈련 과정이 매우 단순하여 머신러닝의 기초 개념을 설명하는 데에 매우 유용하다.

둘째, 딥러닝 심층 신경망 모델 등 대다수의 머신러닝 모델이 훈련 과정에서 선형 회귀 모델의 훈련 방식을 활용한다.

먼저 1장에서 1인당 GDP와 삶의 만족도 사이의 선형 관계를 학습한 선형 회귀 모델이 예측값을 계산하는 방식을 재확인한다.

**예제: 1인당 GDP와 삶의 만족도**

1인당 GDP와 삶의 만족도 사이의 관계를
다음 1차 방정식 함수로 표현하면 다음과 같다.

$$\text{삶의만족도} = \theta_0 + (\text{1인당GDP}) \cdot \theta_1$$

선형 회귀 모델은 1인당 GDP가 주어지면 삶의 만족도를 예측하는 모델을 훈련시키기 위해
위 함수를 활용한다. 

이를 보다 수학적으로 표현하면 다음과 같다. 

$$\hat y = \theta_0 + x_1 \cdot \theta_1$$

위 식에서 $x_1$은 1인당 GDP를,
$\hat y$는 예측된 삶의 만족도를 가리킨다.

**예제: 캘리포니아 주택 가격 예측**

2장에서 활용한 캘리포니아 주택 가격을 예측하는 선형 회귀 모델은
구역별로 주어진 9개의 특성값을 24개로 변환한 다음에 그 지역의 중위 주택 가격을 예측한다.
즉, 1개의 편향과 함께 24개의 가중치를 활용한 아래 모양의 함수를 이용하여 예측값을 계산한다.

$$\hat y = \theta_0 + x_1 \cdot \theta_1 + \cdots + x_{24} \cdot \theta_{24}$$

* $\hat y$: 구역의 예측된 중위 주택 가격
* $x_i$: 구역의 $i$ 번째 특성값(위도, 경도, 중간소득, 가구당 인원 등)
* $\theta_0$: 편향
* $\theta_i$: $i$ 번째 특성에 대한 가중치.

:::{note} 기울기 vs. 가중치

특성이 하나일 때는 기울기 표현이 적절했지만 특성이 2개 이상일 때는 더 이상 적절하지 않으며,
대신 각 특성값에 가해지는 **가중치**<font size='2'>weight</font>라는 표현이 선호된다.
:::

**선형 회귀 예측값**

위 두 예제에서 설명한 선형 회귀 모델이 
예측값을 생성할 때 사용하는 선형 함수를 일반화하면 다음과 같다.

먼저 훈련셋에 포함된 샘플이 $n$ 개의 특성 $x_1$, $x_2$, ..., $x_n$을 갖는다고 가정할 때,
선형 회귀 모델은 아래 식을 이용하여 예측값을 계산한다.
$\theta_0$와 1을 곱해주는 이유는 다른 항과의 형식을 맞추기 위함이다.

$$\hat y = 1\cdot \theta_0 + x_1 \cdot \theta_1 + \cdots + x_n \cdot \theta_{n}$$

아래 이미지는 위 식을 시각화한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/perceptron02.png" width="250"/>
</div>

**선형 회귀 모델의 파라미터: 편향과 가중치**

머신러닝 모델을 훈련하는 주요 목표는 입력값과 타깃 사이에 존재하는 숨은 관계를 찾아내는 것이다.
예를 들어, 선형 회귀 모델은 입력값과 예측값 사이의 관계를 입력 특성과 가중치 $\theta_i$의 선형 조합을 통해 설명한다.
따라서 편향과 가중치는 훈련을 통해 학습해야 하는 파라미터이며, 이는 훈련 데이터셋으로부터 추론해야 하는 정보이다.
선형 회귀 모델을 훈련할 때는 보통 편향 $\theta_0$을 0으로, 나머지 가중치 $\theta_i$는 무작위 값으로 초기화한다.
이후 경사하강법을 적용하여 모든 파라미터(편향과 가중치)를 점차 더 적합한 값으로 업데이트한다.

:::{note} 모델 훈련의 다양성

모든 머신러닝 모델은 선형 회귀뿐만 아니라 각자의 고유한 방식으로 학습한 파라미터를 활용해 예측값을 계산한다.
하지만 머신러닝과 딥러닝의 많은 모델이 선형 회귀 방식을 변형하거나 응용하기 때문에, 그 중요성이 더욱 크다.
:::

### 행렬 연산 표기법

지금까지의 설명은 하나의 입력 샘플에 대해 예측값이 계산되는 과정이다.
그런데 머신러닝 모델은 여러 개의 입력 데이터 샘플에 대해 동시에 예측값을 계산하도록 세팅되어 있다.

실제로 사이킷런의 모든 머신러닝 모델은 입력값으로 2차원 어레이를 요구한다.
예를 들어 4개의 특성을 갖는 입력값에 대해 훈련된 모델은 입력값으로 임의의 양의 정수 `m`에 대해
`(m, 4)` 모양의 데이터프레임 또는 2차원 어레이를 입력값으로 받아 예측값을 계산한다.
이때 `m`은 훈련셋의 크기를 가리키거나 그 일부가 될 수 있다.

아래 예제는 네 개의 특성을 갖는 3개의 샘플로 구성된 입력 데이터셋에 대해
선형 회귀 모델이 파라미터를 이용하여 예측값을 계산하는 과정을 상세히 설명한다.

:::{prf:example} 예측값 행렬 연산
:label: matrix_op_prediction

4개의 특성을 사용하는 데이터 샘플 3개로 구성된 입력 데이터셋으로 훈련하는 `LinearRegression` 모델은 훈련중에 또는 실전에서
아래 `(3, 4)` 모양의 2차원 어레이를 입력 데이터셋으로 받으면 3개의 예측값을 동시에 계산한다.

```python
X_input = array([[0.89, 0.37, 0.39, 0.97],
                [0.95, 0.04, 0.24, 0.02],
                [0.57, 2.91, 1.25, 0.39]])
```

위 입력 데이터셋에 포함된 각각의 샘플 $x^{(1)}$, $x^{(2)}$, $x^{(3)}$에 대한 
예측값 $\hat y_1$,$\hat y_2$, $\hat y_3$을 동시에 계산하기 위해
선형 회귀 모델은 1개의 편향 $\theta_0$와
4개의 가중치 $\theta_1$, $\theta_2$, $\theta_3$, $\theta_4$를 
사용한다.

이제부터 세 개의 입력 샘플에 대한 예측값 3개를 동시에 계산하는 과정을
행렬 연산을 활용하여 설명한다.
먼저 `(3, 4)` 모양의 어레이를 다음과 같이 3x4 모양의 행렬에 해당한다.

- $x_j^{(i)}$: $i$-번째 샘플의 $j$-번째 특성값
- $i$는 1부터 3까지, $j$는 1부터 4까지 움직임.

$$
\begin{bmatrix} 
x_1^{(1)}\,\, x_2^{(1)}\,\, x_3^{(1)}\,\, x_4^{(1)} \\[1ex]
x_1^{(2)}\,\, x_2^{(2)}\,\, x_3^{(2)}\,\, x_4^{(2)} \\[1ex]
x_1^{(3)}\,\, x_2^{(3)}\,\, x_3^{(3)}\,\, x_4^{(3)}
\end{bmatrix}
$$

3개의 예측값 $\hat y_1$,$\hat y_2$, $\hat y_2$을 계산하기 위해 선형 회귀 모델은
아래 형식의 행렬곱셈을 수행한다.
첫째 열에 추가된 1은 편향에 곱해지는 값을 맞추기 위함임에 주의한다.

$$
\begin{align*}
\hat{\mathbf y} 
= \begin{bmatrix}
\hat y_1 \\[1ex]
\hat y_2 \\[1ex]
\hat y_3 
\end{bmatrix}
&= \begin{bmatrix} 
1\,\, x_1^{(1)}\,\, x_2^{(1)}\,\, x_3^{(1)}\,\, x_4^{(1)} \\[1ex]
1\,\, x_1^{(2)}\,\, x_2^{(2)}\,\, x_3^{(2)}\,\, x_4^{(2)} \\[1ex]
1\,\, x_1^{(3)}\,\, x_2^{(3)}\,\, x_3^{(3)}\,\, x_4^{(3)}
\end{bmatrix}
\begin{bmatrix}
\theta_0 \\
\theta_1 \\
\theta_2 \\
\theta_3 \\
\theta_4
\end{bmatrix} \\[8ex]
&= \begin{bmatrix} 
1\cdot \theta_0 + x_1^{(1)}\cdot \theta_1 + x_2^{(1)}\cdot \theta_2 + x_3^{(1)}\cdot \theta_3 + x_4^{(1)}\cdot \theta_4 \\[1ex]
1\cdot \theta_0 + x_1^{(2)}\cdot \theta_1 + x_2^{(2)}\cdot \theta_2 + x_3^{(2)}\cdot \theta_3 + x_4^{(2)}\cdot \theta_4 \\[1ex]
1\cdot \theta_0 + x_1^{(3)}\cdot \theta_1 + x_2^{(3)}\cdot \theta_2 + x_3^{(3)}\cdot \theta_3 + x_4^{(3)}\cdot \theta_4
\end{bmatrix}
\end{align*}
$$

따라서 앞서 언급된 `(3, 4)` 모양의 2차원 어레이 `X_input`에 대한 
선형 회귀 모델의 예측값은 
다음과 같이 단순한 행렬 연산에 의해 계산된다.

$$
\hat{\mathbf y} 
= \begin{bmatrix}
\hat y_1 \\[1ex]
\hat y_2 \\[1ex]
\hat y_3 
\end{bmatrix}
= \begin{bmatrix} 
1\cdot \theta_0 + 0.89\cdot \theta_1 + 0.37\cdot \theta_2 + 0.39\cdot \theta_3 + 0.97\cdot \theta_4 \\[1ex]
1\cdot \theta_0 + 0.95\cdot \theta_1 + 0.04\cdot \theta_2 + 0.24\cdot \theta_3 + 0.02\cdot \theta_4 \\[1ex]
1\cdot \theta_0 + 0.57\cdot \theta_1 + 2.91\cdot \theta_2 + 1.25\cdot \theta_3 + 0.39\cdot \theta_4
\end{bmatrix}
$$
:::

### 머신러닝 모델 훈련의 목표

머신러닝 모델의 훈련은 타깃에 최대한 가까운 예측값 계산을 목표로 한다.
예를 들어, 이전 예제에서 활용한 선형 회귀 모델은 좋은 예측값을 계산하기 위해 
적절한 5개의 파라미터 $\theta_0$, $\theta_1$, $\theta_2$, $\theta_3$, $\theta_4$를
훈련을 통해 학습해야 한다.

**비용 함수**

비용 함수<font size='2'>cost function</font>는 모델의 성능이 얼마나 나쁜가를 계산한다.
비용 함수가 계산하는 값이 작을 수록 해당 모델의 성능이 좋은 것이다.

:::{admonition} 비용 함수와 손실 함수
:class: note

비용 함수는 손실 함수<font size='2'>loss function</font>,
비용 함수에 의해 계산된 값은 손실값<font size='2'>loss</font>이라 부르기도 한다.
:::

**MSE: 회귀 모델의 비용 함수**

비용 함수는 모델 종류와 목표에 따라 다르게 정의되지만
회귀 모델의 경우 일반적으로 **평균 제곱 오차**<font size="2">mean squared error</font>(MSE)를
비용 함수로 사용한다.
선형 회귀 모델의 MSE를 계산하는 수식은 다음과 같다.

$$
\begin{align*}
\mathrm{MSE}(\mathbf{\theta}) &= \frac{(\mathbf{x}^{(1)}\, \mathbf{\theta} - y^{(1)})^2 + (\mathbf{x}^{(2)}\, \mathbf{\theta} - y^{(2)})^2 + \cdots + (\mathbf{x}^{(m)}\, \mathbf{\theta} - y^{(m)})^2}{m} \\[.7ex]
&= \frac 1 m \sum_{i=1}^{m} (\mathbf{x}^{(i)}\, \mathbf{\theta} - y^{(i)} )^2
\end{align*}
$$

위 수식에서 포함된 기호들의 의미는 다음과 같다.

| 기호 | 의미 |
| :---: | :--- |
| $\mathbf{x}^{(i)}$ | $i$ 번째 샘플. 단 0번 인덱스에 1이 추가됨. |
| $y^{(i)}$ | $i$ 번째 샘플에 대한 타깃 |
| $\mathbf{\theta}$ | 파라미터(편향과 가중치)로 구성된 1차원 어레이 $(\theta_0, \theta_1, \dots, \theta_n)$ |
| $m$ | 입력 데이터셋 크기 |

또한 $\mathbf{x}^{(i)}\, \mathbf{\theta}$ 는 $i$-번째 샘플에 대한 예측값 $\hat y^{(i)}$를 가리킨다.
이때 $i$는 $1$부터 $m$까지 움직인다.

$$
\hat y^{(i)} = \mathbf{x}^{(i)}\, \mathbf{\theta} = 1\cdot \theta_0 + x^{(i)}_1 \cdot \theta_1 + \cdots + x^{(i)}_n \cdot \theta_{n}
$$

:::{note}

$\mathrm{MSE}(\mathbf{\theta})$ 계산할 때,
선형 회귀 모델에서는 예측값을 
$\mathbf{x}^{(i)}\, \mathbf{\theta}$로 표현한다.
하지만 이 표현을 일반화하여 $\hat y(\mathbf{x}^{(i)}, \mathbf{\theta})$로 쓰렴,
임의의 모델의 예측값를 나타낼 수 있다.
따라서 임의의 모델에 대한 MSE는 다음과 같이 정의된다.

$$
\mathrm{MSE}(\mathbf{\theta}) = \frac 1 m \sum_{i=1}^{m} (\hat y(\mathbf{x}^{(i)}, \mathbf{\theta}) - y^{(i)} )^2
$$
:::

**모델 훈련의 최종 목표**

회귀 모델의 경우 훈련셋이 주어졌을 때 $\mathrm{MSE}(\mathbf{\theta})$가 최소가 되도록 하는 
$\mathbf{\theta}$를 찾아야 한다.
선형 회귀의 경우 모델에 따라 다음 두 가지 방식 중 하나를 이용하여 해결한다.

* 방식 1: 정규방정식 또는 특이값 분해(SVD)
* 방식 2: 경사하강법(Gradient descent)

정규 방정식은 `LinearRegression` 등 선형 회귀를 활용하는 극히 일부 모델에서, 그것도 훈련셋의 크기와 입력 특성 개수가 모두 작을 때만 활용된다. 
반면에 `SGDRegressor` 등의 모델이 사용하는
경사하강법은 딥러닝 모델에서도 기본으로 활용되는 훈련 기법이다.
이런 의미에서 정규 방정식은 여기서는 다루지 않는다.

(sec:gradient-descent)=
## 경사하강법

경사하강법을 이해하려면 먼저 아래 개념들을 충분히 숙지해야 한다.

**하이퍼파라미터<font size="2">hyperparameter</font>**

훈련시킬 모델을 지정할 때 사용되는 설정 옵션, 
즉 해당 클래스의 객체를 생성할 때 클래스의 생성자 함수에 전달되는 인자들을 가리킨다.
대표적으로 학습률, 에포크, 허용 오차, 배치 크기 등이 있다.

**파라미터**<font size="2">parameter</font>

선형 회귀 모델에 사용되는 편향과 가중치 파라미터처럼 모델 훈련중에 학습되는 값들을 가리킨다.
모델 훈련을 통해 학습된 파라미터는 훈련된 모델 객체의 속성으로 저장된다.

**배치 크기와 배치**

배치 크기<font size='2'>batch size</font>는 모델의 하이퍼파라미터로 지정되는 값으로,
지정된 배치 크기만큼의 샘플 데이터셋에 대해서 모델이 동시에 예측값을 계산한다.
이때 지정된 배치 크기의 샘플 데이터셋을 배치<font size='2'>batch</font>라 부른다.
배치 크기는 모델에 따라 다르게 지정된다.

- 사이킷런 모델: 배치 크기 선택 옵션을 지원하지 않음. 경사하강법을 적용하는 모델의 경우 모델에 따라 배치 크기가 다르게 지정됨.
    - `LinearRegression`: 경사하강법 사용하지 않음.
    - `SGDRegressor`: $m = 1$ (확률적 경사하강법)
    - `LogisticRegression`: 최적화 알고리즘에 따라 다르게 지정됨.
    
- 딥러닝 심층 신경망 모델: 배치 크기 선택 옵션을 제공함. 일반적으로 8, 16, 32, 64, 128, 256 등으로 지정.

**비용 함수**

평균 제곱 오차(MSE)처럼 모델의 성능이 얼마나 나쁜가를 계산하는 함수다.
비용 함수의 값, 즉 손실값은 배치 단위로 계산된다.

:::{prf:example}

회귀 모델의 배치 단위로 계산되는 손실값은 일반적으로 평균 제곱 오차(MSE)로 계산된다.
예를 들어
`SGDRegressor` 모델은 크기가 1인 배치로 훈련하기에
매 스텝에서 하나의 입력 샘플 $\mathbf{x}^{(i)}$에 대해 MSE를 계산한다.

$$
\mathrm{MSE}(\mathbf{\theta}) = 
\big(\mathbf{x}^{(i)}\, \mathbf{\theta} - y^{(i)}\big)^2
$$
:::

**비용 함수의 그레이디언트 벡터**<font size='2'>gradient vector</font>

비용 함수 $\mathrm{MSE}(\mathbf{\theta})$를 $\mathbf{\theta}$에 대해
미분한 결과를 비용 함수의 그레이디언트 벡터라 한다.
이를 $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$로 표기하며,
파라미터 $\mathbf{\theta}$를 어떤 방향과 크기로 조정해야
손실값을 줄일 수 있는지를 알려준다.

여기서는 그레이디언트에 대한 자세한 정의와 설명은 다루지 않는다.
함수 미분과 친숙한 경우 [연습문제 3](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-training_models.ipynb#scrollTo=5MiN6a4E6wKf)번에 도전해 볼 것을 권장한다.

**스텝**<font size='2'>step</font>

스텝은 하나의 배치에 대해 예측값과 손실값(비용)을 계산하고,
이후 손실값을 줄이는 방향으로 파라미터를 한 번 업데이트하는 과정이다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/step.png" width="400"/>
</div>

:::{prf:example}

`SGDRegressor` 모델은 크기가 1인 배치로 훈련하기에 하나의 스텝에 하나의 데이터 샘플만 활용한다.
즉, 아래 과정으로 구성된 스텝을 훈련셋에 포함된 각각의 샘플에 대해 반복해서 진행한다.

- 하나의 데이터 샘플에 대한 예측값 계산
- MSE 계산
- MSE의 그레이디언트 벡터 계산
- $\theta$ 업데이트
:::

**학습률($\eta$)**

훈련 스텝마다 파라미터 $\mathbf{\theta}$를 얼만큼씩 조정할 것인지를 정하는 비율이다.
학습률이 너무 크거나 작으면 모델이 적절한 파라미터를 제대로 학습하지 못할 수 있다.

**에포크와 스텝 횟수**

배치 크기는 1부터 훈련셋 전체의 크기인 $m$ 사이의 정수로 정해진다.
따라서 스텝을 여러번 진행해야 훈련셋 전체에 대한 훈련을 완료할 수 있다.
에포크는 이처럼 여러 번의 스텝을 진행하여 훈련셋 전체를 대상으로
예측과 파라미터 업데이트를 한 번 진행하는 과정이다.
몇 번의 에포크 동안 훈련을 반복할지도 하이퍼파라미터로 지정된다.

스텝 횟수는 에포크 동안 실행된 스텝의 횟수, 즉 파라미터를 조정한 횟수다.
다음이 성립한다.

    스텝 횟수 = (훈련셋 크기) / (배치 크기)

:::{prf:example}

훈련셋 크기가 1,000이고 배치 크기가 10이면 에포크마다 100번의 스텝이 실행된다.
반면에 `SGDRegressor` 모델은 크기가 1인 배치로 훈련하기에 한 번의 에포크 동안 훈련셋 크기만큼의 스텝이 진행된다.
:::

**최적의 모델**

최종적으로 훈련을 통해 얻고자 하는 모델이며,
비용 함수의 값, 즉 손실값을 최소화하는 파라미터를 학습한(찾아낸) 모델이다.

**전역/지역 최소값**

비용 함수의 전역/지역 최소값이다. 
예를 들어 선형 회귀 모델의 경우엔 평균 제곱 오차(MSE) 함수가 갖는 전역/지역 최소값을 가리킨다.

사실 선형 회귀 모델은 전역 최솟값만 가진다.
따라서 적절한 학습률사용하면 언제나 최적의 모델을 훈련시킬 수 있다.
반면에 비선형 모델은 훈련을 통해 반드시 최적의 모델을 학습시킨다는 보장이 없다.
이유는 모델이 훈련을 통해 찾아낸 모델의 파라미터가 반드시 비용함수의 전역 최솟값을 보장하는 파라미터가 아닌
지역 최솟값만 보장할 수 있기 때문이다.

비선형 모델 훈련의 어려움에 대해서는 아래에서 좀 더 설명한다.

**허용 오차**<font size="2">tolerance</font>

모델 훈련은 에포크를 반복하며 진행되지만, 특정 시점 이후로는 에포크를 계속하더라도 성능 개선이 더 이상 좋아지지 않을 수 있다.
이는 파라미터를 얼만큼 업데이트할지를 결정할 때 사용되는 비용 함수의 그레이디언트 벡터의 크기가 너무 작아졌기 때문이다.
이런 경우엔 훈련을 종료시킬 필요가 있으며, 이때 허용 오차가 손실값 변화 정도의 기준값으로 사용된다.
그레이디언트 벡터의 크기가 파라미터 업데이트에 영향을 미치는 방식은 이어지는 절에서 자세히 설명한다.

### 선형 회귀 모델의 경사하강법

MSE를 비용 함수로 사용하는 선형 회귀 모델의 파라미터를 업데이트하는 스텝에서
사용되는 경사하강법은 아래 과정으로 구성된다.

1. 파라미터 벡터 $\mathbf{\theta}$를 0 또는 임의의 값으로 초기화한 후에 훈련을 시작한다.

1. 지정된 에포크만큼 또는 그레이디언트 벡터
    $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$가 충분히 작아질 때까지 
    아래 과정으로 구성된 훈련 스텝을 반복한다.

    * 하나의 배치에 대해 예측값 생성 후 손실값 $\mathrm{MSE}(\mathbf{\theta})$ 계산.
    * $\mathbf{\theta}$를 아래 점화식을 이용하여 업데이트:

    $$
    \theta^{(\text{new})} = \theta^{(\text{old})}\, -\, \eta\cdot \nabla_\theta \textrm{MSE}(\theta^{(\text{old})})
    $$

    위 식에서 $\eta$는 학습률, 
    $\theta^{(\text{old})}$는 이전 스텝을 통해 얻어진 파라미터 벡터, 
    $\theta^{(\text{new})}$는 업데이트된 파라미터 벡터를 가리킨다.


:::{note} 그레이디언트 벡터의 방향과 크기

그레이디언트 벡터 $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$가 가리키는 방향의
**반대 방향**으로 움직이면 빠르게 $\textrm{MSE}(\mathbf{\theta})$가 
전역 최소값을 갖는 $\mathbf{\theta}$ 접근한다.

아래 이미지는 선형회귀 모델의 손실 함수에 대한 가중치 업데이트를 진행하는 과정을 보여준다.
이해를 위해 먼저 파라미터 벡터 $\mathbf{\theta}$를 벡터가 아닌 하나의 가중치 값으로 간주하면
MSE가 $\mathbf{\theta}$에 대한 2차 함수로 표현됨에 주의한다.

따라서 비용 함수 그래프상의 한 점에서의 기울기는 비용 함수의 $\mathbf{\theta}$에 대한 미분값으로
계산되는 데 이 값이 바로 그레이디언트 벡터에 해당한다. 
그리고 기울기가 양수(또는 음수)이면 기존의 가중치(weight)를 왼쪽(또는 오른쪽)으로 움직여야 비용 함수의 전역 최소값에 가까워진다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/gradient01b.png" width="500"/>
</div>

<p><div style="text-align: center">&lt;이미지 출처: <a href="https://www.analyticsvidhya.com/blog/2020/10/how-does-the-gradient-descent-algorithm-work-in-machine-learning/">Analytics Vidhya</a>&gt;</div></p>

아래 두 이미지는 산에서 가장 경사가 급한 길을 따를 때 가장 빠르게 하산한다는 원리를 보여준다.
이유는 해당 지점에서 그레이디언트 벡터를 계산하면 정상으로 가는 가장 빠른 길을 안내하기에
그 반대방향으로 움직여야 최대한 빠르게 최저점(평지)으로 하산할 수 있기 때문이다.
이미지에서 보여지는 여러 경로는 경사하강법 알고리즘에 따른 다른 경로를 보여준다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/saddle_point_evaluation_optimizers.gif" width="500"/>
</div>
:::

### 학습률의 중요성

선형 회귀 모델은 적절한 학습률로 경사하강법으로 적용할 경우
빠른 시간에 비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 수렴한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-01.png" width="500"/>
</div>

반면에 학습률이 너무 작거나 너무 크면 비용 함수가 전역 최소값을 갖도록 하는 파라미터에
너무 느리게 수렴하거나 아예 수렴하지 않을 수도 있다.

**학습률이 너무 작은 경우**

비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 너무 느리게 수렴한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-02.png" width="500"/>
</div>

**학습률이 너무 큰 경우**

비용 함수가 전역 최소값을 갖도록 하는 $\hat{\theta}$ 에 수렴하지 않고 발산한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-03.png" width="500"/>
</div>

아래 세 이미지는 학습률에 따라 선형 회귀 모델이 최적의 모델로 수렴하는지 여부와 수렴 속도가 달라질 수 있음을 보여준다.
단, 여기서 예제로 사용하는 학습률의 적절성은 모델과 훈련셋에 따라 달라질 수 있음에 주의한다.
보다 상세한 내용은
[(코드 워크아웃) 모델 훈련](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-training_models.ipynb)를 참고한다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-04b.png" width="700"/></div>

| 학습률 (η)  | 설명               | 수렴 속도 |
| :--------: |--------------------| ------ |
| 0.02       | 학습률이 너무 작은 경우 | 매우 느리게 수렴. 많은 에포크 훈련 필요 |
| 0.1        | 학습률이 적절한 경우   | 빠르게 수렴 |
| 0.5        | 학습률이 너무 큰 경우 | 수렴하지 못하고 발산. 훈련할 수록 성능이 나빠짐 |

**비선형 모델 훈련의 어려움**

선형 회귀 모델의 손실함수는 $\theta$대한 2차함수 포물선 그래프로 그려지기에
학습률을 적절하게 잡으면 언제나 최적의 모델로 수렴한다.

반면에 비선형 모델의 경우엔 학습률과 상관 없이 파라미터를 초기화하는 방식에 따라 
지역 최소값에 수렴하거나 수렴하지 못하고 
지역 최솟값으로 수렴하거나 아니면 더 이상의 학습이 잘 되지 않고 정체할 수도 있음을
아래 이미지가 보여준다.

<p><div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-04.png" width="500"/></div></p>

**특성 스케일링의 중요성**

특성들의 스켈일을 통일시키면 보다 빠른 학습이 이루어지는 이유를 아래 두 이미지가 비교해서 보여준다.
밝아질 수록 낮은 지점(작은 손실값)을 의미한다.

왼쪽 이미지는 두 특성의 스케일이 동일하게 조정된 경우엔 비용 함수의 최소값으로 최단거리로 수렴함을 보여준다.
비용 등고선이 두 가중치 파라미터에 대해 원 모양으로 그려지는 경우를 생각하면 된다.

반면에 오른쪽 이미지에서는 두 특성의 스케일이 달라서 비용 함수의 최소값으로 보다 먼 거리를 거쳐서 상대적으로 느리게 수렴한다.
이유는 비용 등고선이 두 가중치 파라미터에 타원 또는 찌그러진 모양으로 그려지기 때문이다.
이를 해결하기 위해서는 전처리 과정에서 데이터를 스케일링해서 특성별 크기를 가능한한 통일시켜야 한다.

<div align="center">
     <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-04a.png" width="500"/>
</div>

### 경사하강법 종류

모델 훈련에 사용되는 경사하강법은
배치 크기 하이퍼파라미터에 따라 세 종류로 나뉜다.

**배치 경사하강법**

배치 크기가 전체 훈련셋의 크기와 같은 경우이다.
에포크마다 한 번의 스텝만 진행된다.

파라미터 업데이트를 에포크에 한 번만 수행 하기에 모델을 지정할 때 에포크를 크게 잡아야 한다.
그렇지 않으면 훈련이 제대로 진행하지 않을 수 있다.
그런데 훈련셋이 크면 그레이디언트를 계산하는 데에 많은 시간과 메모리가 필요해지는 문제가 발생할 수 있다.
이와 같은 이유로 배치 경사하강법을 지원하는 모델은 별로 없다.

**확률적 경사하강법(SGD)**

배치 크기가 1인 경상하강법을 가리킨다.
즉, 매 스텝마다 하나의 훈련 셈플에 대한 예측값 계산과 파라미터 업데이트가 이뤄진다.

스텝에서 사용되는 샘플은 무작위로 선택되며,
모든 샘플이 한 번씩 선택되면 하나의 에포크가 끝난다.
이후 새 에포크가 시작되기 전에 훈련셋이 무작위로 섞여서 
이전 에포크에서와는 다른 순서로 훈련 샘플이 무작위로 선택된다.
이는 모델에게 보다 다양한 훈련을 체험하도록 유도하기 위함이다.

확률적 경사하강법의 장점은 계산량이 매우 적다는 점이다. 
따라서 아주 큰 훈련셋을 이용하여 훈련할 수 있다.
또한 파라미터 조정이 불안정하게 이뤄질 수 있기 때문에 지역 최소값에 상대적으로 덜 민감하다.

반면에 동일한 이유로 경우에 따라 전역 최소값에 수렴하지 못하고 아래 이미지에서처럼 주변을 맴돌 수도 있다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-04c.png" width="300"/>
</div>

요동치는 파라미터를 제어하기 위해 학습률을 학습 과정 동안 천천히 줄어들도록 하는
학습 스케줄<font size="2">learning schedule</font>을 사용한다.
학습 스케줄은 일반적으로 훈련 에포크가 진행될 수록 학습률이 조금씩 작아지도록 지정하는 기법이다.

:::{prf:example} 사이킷런의 `SGDRegressor` 모델
:label: exp-sgdregressor

확률적 경사하강법을 사용하는 선형 회귀 모델이다.
아래 코드는 에포크 크기, 허용 호차, 학습 스케줄, 규제 적용 여부를 지정하는 하이퍼파라미터를 함께
사용하는 것을 보여준다.

```python
SGDRegressor(penalty=l2, 
             max_iter=1000, 
             tol=1e-3,
             learning_rate='invscaling', 
             eta0=0.01,
             power_t=0.25,
             early_stoppingbool=False,
             n_iter_no_change=5, 
             random_state=42)
```

| 파라미터   | 설명  |
|----------|-------|
| `penalty=l2` | 규제 종류. `None`은 규제 미사용 (추후 설명) |
| `max_iter=1000` | 최대 에포크 수                        |
| `tol=1e-3`      | 허용 오차. 손실값이 `n_iter_no_change` 에포크 동안 허용 오차보다 작게 변할 때 훈련 중지.<br> - `early_stoppingbool=False`이면 훈련셋 손실값 사용<br> - `early_stoppingbool=True`이면 검증셋 손실값 사용 |
| `early_stoppingbool=False` | 조기 종료 여부 지정. 기본값은 `False` (추후 설명)                  |
| `learning_rate='invscaling'` | 학습률 조정 방식. `eta = eta0 / pow(t, power_t)`                 |
| `eta0=0.01`  | 초기 학습률. 학습 스케줄에 활용됨   |
| `power_t=0.25` | inverse scaling 방식으로 학습률 조정에 사용되는 지수  |
| `n_iter_no_change=5` | 지정된 에포크 동안 `loss > best_loss - tol` 상태가 유지되면 훈련 중지 |

머신러닝 모델은 언급된 하이퍼파라미터 이외에도 많은 다양한 종류의 모델 설정 하이퍼파라미터를 지원한다.
각 모델별로 사용되는 하이퍼파라미터를 확인하려면 해당 모델의 공식 문서를 참고해야 한다.
:::

**미니 배치 경사하강법**

배치 크기를 2 이상으로 잡는 경사하강법이다. 
보통 2에서 수백 사이로 정한다.
배치 크기를 적절히 크게 잡으면 확률적 경사하강법(SGD) 보다 파라미터의 움직임이 덜 불규칙적이 되며,
배치 경사하강법보다는 훨씬 빠르게 최적 학습 모델에 수렴한다.
다만 SGD에 비해 지역 최소값에 수렴할 위험도가 보다 커질 수 있지만 대부분의 딥러닝 심층 신경망 모델을
훈련할 때 기본적으로 활용다.

**경사하강법 비교**

아래 이미지는 두 개의 가중치 파라미터를 학습시키기 위해 
앞서 설명한 배치/확률적/미니 배치 경사하강법을 진행할 때 학습되는 파라미터들의 변화 과정을 보여준다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-05.png" width="500"/>
</div>


| 알고리즘            | 배치 크기   | 훈련 속도 | 모델 수렴      | 빅데이터 지원 |
| :----------------:| :--------: | :------: | :-----------: | :---------: |
| 배치 경사하강법     | 훈련셋 전체  | 매우 느림 | 정확히 수렴     | 불가능       |
| 확률적 경사하강법    | 1          | 매우 빠름 | 심한 진동      | 가능         |
| 미니 배치 경사하강법 | 2 이상    | 빠름     | 어느 정도 안정적 | 가능        |

(sec:poly_reg)=
## 비선형 데이터 학습: 다항 회귀

**다항 회귀**<font size="2">polynomial regression</font>는
비선형 데이터를 선형 회귀를 이용하여 학습하는 기법이다.

**예제: 2차 함수 모델를 따르는 데이터셋에 선형 회귀 모델 적용**

아래 이미지는 2차 함수의 그래프 형식으로 분포된 데이터셋을 선형 회귀 모델로 학습시킨 결과를 보여준다.
즉, 예측값이 $x_1$에 대한 아래 1차 함수로 계산된다.

$$\hat y = \theta_0 + \theta_1\, x_1$$

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-06.png" width="500"/>
</div>

**예제: 2차 함수 모델를 따르는 데이터셋에 2차 다항 회귀 모델 적용**

반면에 아래 이미지는 $x_1^2$ 에 해당하는 특성을 새로이 추가한 후에
선형 회귀 모델을 학습시킨 결과를 보여준다.
예측값은 $x_1$에 대한 아래 2차 함수 형식으로 계산된다.

$$\hat y = \theta_0 + \theta_1\, x_1 + \theta_2\, x_{1}^2$$

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-07.png" width="500"/>
</div>

**다항 회귀의 단점**

몇차 다항 회귀를 사용해야 할지 일반적으로 알 수 없다. 
또한 심층 신경망처럼 비선형 데이터를 분석하는 보다 좋은 모델이 개발되어 굳이 다항 회귀를 사용할 필요가 없어졌다.
여기서는 비선형 데이터 분석을 선형 회귀 모델로 제대로 예측할 수 없음을 보여주기 위해 언급되었다.

## 과소/과대 적합

사용되는 모델에 따라 훈련된 모델의 성능이 많이 다를 수 있다.
아래 이미지는 기본 선형 모델은 성능이 너무 좋지 않은 반면에
300차 다항 회귀 모델은 너무 과하게 훈련 데이터에 민감하게 반응하는 것을 보여준다.
반면에 2차 다항 회귀 모델이 적절(?)하게 예측값을 계산하는 것으로 보인다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-08.png" width="500"/></div>

**모델 성능 평가: 교차 검증**

일반적으로 어떤 모델이 가장 좋은지 미리 알 수 없다. 
따라서 보통 다양한 모델을 대상으로 교차 검증을 진행하여 성능을 평가한다.
교차 검증 결과에 따른 모델 평가는 다음 두 종류로 나뉜다.

* 과소적합: 훈련 점수와 교차 검증 점수 모두 낮은 경우
* 과대적합: 훈련 점수는 높지만 교차 검증 점수가 상대적으로 많이 낮은 경우

**과소 적합 모델 개선법**

모델이 데이터셋에 과소 적합 하는 일반적인 경우와 개선법은 다음과 같다.

- 너무 단순한 모델 활용: 보다 복잡한 데이터를 학습할 수 있는 모델 활용
- 너무 적은 데이터: 훈련셋 양 늘리기

**과대 적합 모델 개선법**

- 모델 규제 적용: 모델이 훈련중에 데이터에 너무 민감하게 반응하지 않도록 규제 가함. 이어지는 내용 참고
- 보다 많은 데이터: 데이터가 많을 수록 일반적으로 과대 적합이 덜 발생함.

**모델의 일반화 성능**

훈련 과정에서 다루지 않은 새로운 데이터 대한 예측 능력이 모델의 **일반화 성능**이다.
새로운 데이터에 대한 모델의 예측에 나쁜 영향을 미치는 요소는 일반적으로 다음 세 가지가 있다.

- 편향: 실제로는 2차원 모델인데 1차원 모델을 사용하는 경우처럼 데이터의 분포에 대한 잘못된 가정으로 인해 발생한다.
    과소 적합이 발생할 가능성이 매우 높다.

- 분산: 모델이 훈련 데이터에 민감하게 반응하는 정도를 가리킨다.
    고차 다항 회귀 모델처럼 모델이 학습해야하는 파라미터의 수가 많을 수록 분산이 커진다.
    
- 제거 불가능 오류: 노이즈(noise) 등 데이터 자체의 한계로 인해 발생한다.
    데이터 전처리 과정에서 노이즈를 제거해야만 오류를 줄일 수 있다.

**모델 자유도**

모델이 학습해야 하는 파라미터의 수를 모델의 **자유도**<font size='2'>degree of freedom</font>라 부르기도 한다.
자유도가 높을 수록 복잡한 방식으로 예측값을 계산하고, 일반적으로 분산이 크다.

**편향-분산 트레이드오프**

복잡한 모델일 수록 편향을 줄어들지만 분산은 커지는 현상을 가리킨다.

## 모델 규제

훈련 중에 과소 적합이 발생하면 보다 복잡한 모델을 선택해야 한다.
반면에 과대 적합이 발생할 경우 보다 단순한 모델을 사용하거나 모델에 규제를 가해서
모델의 분산을 줄여 과대 적합을 방지하거나 과대 적합이 최대한 늦게 발생하도록 유도해야 한다. 

회귀 모델에 대한 **규제**<font size='2'>regularization</font>는 가중치의 역할을 제한하는 방식으로 이루어지며,
방식에 따라 다음 세 가지 회귀 모델이 지정된다.

* 릿지 회귀
* 라쏘 회귀
* 엘라스틱 넷

**릿지 회귀<font size='2'>Ridge Regression</font>**

아래 비용 함수를 사용한다.

$$J(\theta) = \textrm{MSE}(\theta) + \frac{\alpha}{m} \sum_{i=1}^{n}\theta_i^2$$

* $\theta_0$: 규제에서 제외.
* $m$: 배치 크기
* $\alpha$(알파): 규제 강도. $\alpha=0$일 때 규제 없음.
    - $\alpha$ 가 커질 수록 가중치의 역할이 줄어듦.
        비용을 줄이기 위해 가중치 $\theta_i$를 작게 유지하도록 훈련되어 결국 모델의 분산 정도가 작아짐.

:::{tip}

`StandardScaler` 등을 사용하여 특성 스케일링을 진행 한 다음에
규제를 적용해야 모델의 성능이 좋아진다.
이유는 $\theta_i$가 특성의 크기에 의존하기에
모든 특성의 크기를 비슷하게 맞추면 $\theta_i$가 
보다 일정하게 수렴하기 때문이다.
:::

아래 이미지는 서로 다른 규제 강도를 사용한 릿지 회귀 모델의 훈련 결과를 보여준다.

- 왼쪽: 선형 회귀 모델에 세 개의 $\alpha$ 값 적용.
- 오른쪽: 10차 다항 회귀 모델에 세 개의 $\alpha$ 값 적용.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/ridge01.png" width="600"/>
</div>

**라쏘 회귀<font size='2'>Lasso Regression</font>**

아래 비용 함수를 사용한다.

$$J(\theta) = \textrm{MSE}(\theta) + 2\alpha \, \sum_{i=1}^{n}\mid \theta_i\mid$$

별로 중요하지 않은 특성에 대해 $\theta_i$가 
빠르게 0에 수렴하도록 훈련 중에 유도한다.
이유는 $\mid \theta_i \mid$ 의 미분값이 1 또는 -1 이라는 상대적으로 큰 값이기에
파라미터 업데이트 과정에서 보다 작은 $\mid \theta_i \mid$가 보다 빠르게 0에 수렴하기 때문이다.

아래 이미지는 서로 다른 규제 강도를 사용한 라쏘 회귀 모델의 훈련 결과를 보여준다.

- 왼쪽: 선형 회귀 모델에 세 개의 $\alpha$ 값 적용.
- 오른쪽: 10차 다항 회귀 모델에 세 개의 $\alpha$ 값 적용.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/lasso01.png" width="600"/>
</div>

**엘라스틱 넷<font size='2'>Elastic Net</font>**

릿지 회귀와 라쏘 회귀를 절충한 아래 비용 함수를 사용한다.

$$
J(\theta) = 
\textrm{MSE}(\theta) + 
r\cdot \bigg (2 \alpha \, \sum_{i=1}^{n}\mid\theta_i\mid \bigg) + 
(1-r)\cdot \bigg (\frac{\alpha}{m}\, \sum_{i=1}^{n}\theta_i^2 \bigg )
$$

- $r$: 릿지 회귀에 사용되는 규제와 라쏘 회귀에 사용되는 규제의 사용 비율
- 규제 강도를 의미하는 $\alpha$가 각 규제에 가해지는 정도가 다름에 주의할 것.

:::{tip} 🎯 규제 선택 가이드 (L1 vs L2)

라쏘 회귀와 릿지 회귀에서 사용된 규제는 각각 **L1 규제**와 **L2 규제**라 부른다.
L1 규제와 L2 규제는 선현 회귀 모델 뿐만 아니라
많은 회귀 모델에서 함께 활용되며, 아래 기준으로 선택할 것을 추천한다.

* 아무 정보가 없다면 보통 L2 규제를 먼저 시도하는 것이 안정적이다.
* 일부 특성만 중요하다고 판단되면, 불필요한 특성의 가중치를 완전히 0으로 만들어주는 라쏘 규제(L1 규제)가 효과적이다.
* 상관관계가 높은 특성이 많거나, 전체 특성 수가 훈련 셋의 샘플 수보다 많다면 엘라스틱 넷 형식의 L1 규제와 L2 규제의 혼합이 추천된다.
:::

(sec:early-stopping)=
## 조기 종료

**조기 종료**<font size='2'>early stopping</font>는 
모델이 훈련셋에 과대 적합하는 것을 방지하기 위해 훈련을 적절한 시기에 중단시키는 기법이며,
가장 많이 사용된다.
조기 종료는 검증셋에 대한 비용 함수의 값, 즉 검증셋에 대한 손실값이 더 이상 제대로 줄어들지 않으면 바로 훈련을 종료한다.

아래 그래프는 2차 함수 곡선 형식으로 분포된 데이터셋에 90차 다항 회귀 모델을 훈련시킨 결과를 보여준다.
실행된 에포크가 많아질 수록 훈련셋에 대한 모델의 비용(RMSE)이 점차 낮아지는 반면에
검증셋에 대한 비용은 250 에포크 정도 지나면서 늘기 시작한다. 
즉, 모델이 훈련셋에 과하게 적응하기 시작했고, 이는 모델의 일반화 성능이 떨어지기 시작함을 의미한다.
따라서 허용된 최대 500 에포크를 훈련하지 않고 250 에포크 정도에서 훈련을 멈추도록 하는 게
조기 종료다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-11.png" width="500"/>
</div>

확률적 경사하강법, 미니 배치 경사하강법에서는 비용 함수의 값, 즉 손실값이 보다 많이 진동하기에
비용이 언제 최소가 되었는지 알기 어렵다.
따라서 한동안, 보통 5 에포크 정도, 손실값이 개선되지 않을 때 훈련을 멈추고 그동한 
학습된 최적의 파라미터를 사용하는 모델로 되돌리는 게 좋다.

사이킷런의 `SGDRegressor` 모델을 포함한 많은 모델이 조기 종료 기능을 지원한다.
`SGDRegressor`의 경우 위에서 언급한 `early_stopping` 하이퍼파라미터의 값을 `True`로 지정하면
훈련셋의 일부를 검증셋으로 진행한 다음에 조기 종료 기능을 갖춘 상태로 훈련을 진행한다.

## 로지스틱 회귀

로지스틱 회귀는 회귀 모델의 결과를 분류 모델로 활용할 수 있도록 해주며
분류 모델에서 가장 중요한 역할을 수행한다.
로지스틱 회귀는 이진 분류에 사용되며, 다중 클래스 분류에는 로지스틱 회귀을 일반화한 소프트맥스 회귀가 사용된다.

### 시그모이드 함수

로지스틱 회귀는 선형 회귀 방식으로 계산된 예측값을 시그모이드 함수에 적용하여 0과 1 사이로 변환된 값을
모델의 최종 예측값으로 활용한다. 

시그모이드<font size='2'>sigmoid</font> 함수는 다음과 같이 정의된다.

$$\sigma(t) = \frac{1}{1 + e^{-t}}$$

그래프로 그리면 $t=0$일 때 0.5를 갖고, $t$가 양의 무한대로 커질 수록 1에 수렴하고
음의 무한대로 커지면 0에 수렴한다.

<p><div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-12.png" width="500"/></div></p>

**확률 예측**

로지스티 회귀 모델은 먼저 선형 회귀 모델이 예측한 값에 시그모이드 함수를
적용하여 0과 1 사이의 값, 즉 양성일 **확률** $\hat p$ 를 계산한다.

$$
\hat p = h_\theta(\mathbf{x}) 
= \sigma(\theta_0 + \theta_1\, x_1 + \cdots + \theta_n\, x_n)
$$

**예측값**

로지스틱 회귀 모델의 예측값은 계산된 확률이 0.5 이상인지 여부로 결정한다.

$$
\hat y = 
\begin{cases}
0 & \text{if}\,\, \hat p < 0.5 \\[1ex]
1 & \text{if}\,\, \hat p \ge 0.5
\end{cases}
$$

이는 다음과 같이 가중치와 특성의 선형 조합 결과가 0 이상인지 여부에 따라 양성 또는 음성으로 판별함을 의미한다.

* 양성: $\theta_0 + \theta_1\, x_1 + \cdots + \theta_n\, x_n \ge 0$ 인 경우
* 음성: $\theta_0 + \theta_1\, x_1 + \cdots + \theta_n\, x_n < 0$ 인 경우

### 로그 손실 함수

로지스틱 회귀 모델은 
아래 **로그 손실**<font size='2'>log loss</font> 함수를 비용 함수로 사용한다.

$$
J(\theta) = 
- \frac{1}{m}\, \sum_{i=1}^{m}\, \left( y^{(i)} \cdot \log(\,\hat p^{(i)}\,) + (1-y^{(i)}) \cdot \log(\,1 - \hat p^{(i)}\,)\right)
$$

그러면 양성 샘플에 대해서는 1에 가까운 확률값을,
음성 샘플에 대해서는 0에 가까운 확률값을 내도록 훈련한다.

**로그 손실 함수 이해**

틀린 예측을 하면 로그 손실값이 매우 커진다.
예를 들어 하나의 입력 샘플에 대한 손실값은 다음과 같다.

$$
-\left(y \cdot \log(\,\hat p\,) + (1-y) \cdot \log(\,1 - \hat p\right)
$$

아래 이미지는 예측을 틀리게 할 때 위 손실값이 무한히 커질 수 있음을 보여준다.

<p><div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-12-10a.png" width="500"/></div></p>

| 실제 라벨         | 로그 손실 설명 |
|------------------|----------------|
| 1 (양성) | 예측 확률 $\hat p$가 0에 가까워질수록, 즉 더 많이 틀리게 예측할수록<br> 손실값 $-\log(\hat{p})$가 무한히 커짐 |
| 0 (음성) | 예측 확률 $\hat p$가 1에 가까워질수록, 즉 더 많이 틀리게 예측할수록<br> 손실값 $-\log(1 - \hat{p})$가 무한히 커짐 |

### 붓꽃 이진 분류

붓꽃의 품종 분류를 로지스틱 회귀로 진행한다.
붓꽃 데이터셋은 머신러닝 분류 모델을 소개할 때 자주 활용되는 유명한 데이터셋이다.

**붓꽃 데이터셋 기초 정보**

붓꽃 데이터셋의 샘플은 꽃받침<font size='2'>sepal</font>의 길이와 너비, 
꽃입<font size='2'>petal</font>의 길이와 너비 등 총 4개의 특성으로 
이루어진다. 

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/datapy/master/jupyter-book//images/iris_petal-sepal.png" width="500"/>
</div>

라벨은 0, 1, 2 중에 하나이며 각 숫자는 하나의 품종을 가리킨다. 

| 라벨 | 종 이름 (영문)     | 종 이름 (한글) |
|------|-------------------|----------------|
| 0    | Iris-Setosa       | 세토사         |
| 1    | Iris-Versicolor   | 버시컬러       |
| 2    | Iris-Virginica    | 버지니카       |

훈련셋의 첫 5개 샘플은 다음과 같다.

```python
    sepal length (cm) sepal width (cm) petal length (cm) petal width (cm)
0    5.1               3.5              1.4               0.2
1    4.9               3.0              1.4               0.2
2    4.7               3.2              1.3               0.2
3    4.6               3.1              1.5               0.2
4    5.0               3.6              1.4               0.2
```

**이진 분류: 버지니카 품종 감지기**

꽃잎의 길이와 너비 두 특성을 이용하여 붓꽃의 품종이
버지니카인지 여부를 판별하는 이진 분류기를
로지스틱 회귀 모델로 훈련시킨다.

훈련셋과 타깃셋은 다음과 같이 구분되어 있다.

| 구분 | 변수이름 | 설명 |
| :--- | :--- | :--- |
| 훈련셋 | `X` | 꽃잎의 길이와 너비 특성만 포함하는 붓꽃 데이터 샘플<br> 150개로 구성된 넘파이 2차원 어레이 (모양: `(150, 2)`) |
| 타깃 데이터셋 | `y` | 0(버지니카 아님)과 1(버지니카 품종)로 구성된 넘파이<br> 1차원 어레이 (모양: `(150, )`) |


**로지스틱 회귀 모델 훈련**

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

log_reg = LogisticRegression(l1_ratio=0.0,
                             C=2,
                             random_state=42)

log_reg.fit(X_train, y_train)
```

**로지스틱 회귀 모델 규제**

`LogisticRegression` 모델의 하이퍼파라미터 `penalty` 와 `C` 를 이용하여 규제와 규제의 강도를 지정한다. 

| 하이퍼파라미터      | 기능               |
|------------------|-------------------|
| `l1_ratio=0.0`   | `l1_ratio * L1 + (1 - l1_ratio) * L2` 형식의 규제 적용.<br>기본값 0은 L2 규제, 1은 L1 규제, 0과 1 사이 값은 Elastic Net 규제 |
| `C=1`              | `solver`로 지정된 알고리즘에 따라 L2 또는 L1 규제에 사용되는 $\alpha$ 값의 역수.<br>값이 0에 가까울수록 강한 규제 의미 |

**`predict()` vs `predict_proba()`**

`LogisticRegression`, `RandomForestClassifier`을 포함한 사이킷런의 대부분의 분류기는 
범주를 예측하는 `predict()`  메서드 이외에
범주별로 입력 샘플이 속할 확률을 계산하는 `predict_proba()` 메서드를 지원한다.
단, `SGDClassifer` 클래스는 기본적으로 확률 예측은 지원하지 않는다.

아래 코드는 테스트셋의 첫 5개 샘플에 대한 버지니카 품종 여부를 판별한다.

```python
log_reg.predict(X_test[:5])
```

결과는 다음과 같다.
이진 분류기의 예측값은 양성 또는 음성이다.
따라서 테스트셋의 셋째 샘플만 버지니카 품종임을 알려준다.

```python
array([False, False,  True, False, False])
```

반면에 아래 코드는 테스트셋의 첫 5개 샘플에 대한 범주별 확률을 계산한다.

```python
log_reg.predict_proba(X_test[:5])
```

반환값은 샘플별 음성일 확률과 양성일 확률을 담은 2차원 어레이다.
테스트셋의 셋째 샘플의 경우에만 양성일 확률, 즉 버지니카 품종일 확률이 압도적으로 높다.

```python
array([[8.54331954e-01, 1.45668046e-01],
       [9.99998452e-01, 1.54794895e-06],
       [3.58237243e-04, 9.99641763e-01],
       [8.27775210e-01, 1.72224790e-01],
       [7.15555507e-01, 2.84444493e-01]])
```

(sec:softmax-regression)=
## 소프트맥스 회귀

로지스틱 회귀 모델을 일반화하여 다중 클래스 분류를 지원하도록 만든 모델이
**소프트맥스 회귀**<font size='2'>Softmax regression</font>다.
즉, 소프트맥스 회귀는 3개 이상의 범주로 분류하는 다중 클래스 분류 모델이다. 

소프트맥스 회귀 모델은 $K$ 개의 범주 각각에 대해 주어진 데이터 샘플이 해당 클래스에
포함될 확률을 계산한 다음에 가장 큰 확률값을 갖는 범주를 최종 예측값으로 지정한다.

주어진 데이터 샘플에 특정 범주에 속할 확률은 **소프트맥스 점수**를 이용하여 계산된다.

### 소프트맥스 점수

데이터 샘플 $\mathbf x = [x_1, \dots, x_n]$가 주어졌을 때 
$k$-범주에 대한 소프트맥스 점수 $s_k(\mathbf x)$ 는
선형 회귀 방식으로 계산한다.

$$
s_k(\mathbf{x}) = \theta_0^{(k)} + \theta_1^{(k)} x_1 + \cdots + \theta_n^{(k)} x_n
$$    

위 식에서 $\theta_i^{(k)}$ 는 $k$-범주에 대한 소프트맥스 점수를
선형 회귀 방식으로 계산하기 위해 필요한 $i$ 번째 특성에 대한 가중치 파라미터를 가리킨다.
따라서 $K$ 개의 클래스로 분류하는 소프트맥스 회귀는 
아래에 언급된 총 $K \cdot (n+1)$ 개의 파라미터를 학습시켜야 하며 그만큼 훈련시간이 오래 걸린다.

$$
\begin{align*}
\theta^{(1)} &= (\theta_0^{(1)}, \theta_1^{(1)}, \cdots, \theta_n^{(1)}) \\
& \,\,\,\vdots \\[1ex]
\theta^{(K)} &= (\theta_0^{(K)}, \theta_1^{(K)}, \cdots, \theta_n^{(K)}) \\
\end{align*}
$$

### 소프트맥스 함수

소프트맥스 함수는
$K$ 개의 범주 각각에 대해 계산된 소프트맥스 점수를 통합하여 
각각의 범주에 포함될 확률을 계산한다.
이때 계산된 $K$ 개의 확률의 합은 1이 되도록 한다.
소프트맥스 함수는 결국 계산된 $K$ 개의 소프트맥스 점수를
$K$ 개의 상대적 확률값으로 변환한다.

앞서 계산된 $K$ 개의 소프트맥스 점수를 이용하여 해당 샘플이 
$k$-번째 범주에 속할 확률 $\hat p_k$는 다음과 같이 계산된다.

$$
\begin{align*}
\hat p_k &= \frac{\exp(s_k(\mathbf x))}{\exp(s_1(\mathbf x)) + \cdots + \exp(s_K(\mathbf x))} \\[3.5ex]
&= \frac{\exp(s_k(\mathbf x))}{\sum\limits_{j=1}^{K}\exp(s_j(\mathbf x))}
\end{align*}
$$

**소프트맥스 회귀 모델의 예측값**

소프트맥스 회귀 모델의 각 샘플에 대한 최종 예측값(라벨) $\hat y$는
예측 확률값 $\hat p_k$가 최댓값이 되는 $k$로 지정된다.
즉, 다음이 성립한다.

$$
\hat y = k  \quad\Longleftrightarrow\quad \hat p_k = \max([\hat p_1, \hat p_1, \dots, \hat p_{K}])
$$

### 크로스 엔트로피

$k$-범주 각각에 대한 적절한 가중치들의 벡터 $\mathbf{\theta}^{(k)} = [\theta_0^{(k)}, \theta_1^{(k)}, \dots, \theta_n^{(k)}]$를 
경사하강법을 이용하여 업데이트 한다.
이를 위해 예측 라벨이 아닌 범주별 예측 확률값에 근거한 **크로스 엔트로피**<font size='2'>cross entropy</font>를 소프트맥스 회귀 모델의 비용 함수로 사용한다.

$$
\begin{align*}
J(\Theta) &= - \frac{1}{m}\, \sum_{i=1}^{m} \big( y^{(i)}_1\, \log ( \hat{p}_1^{(i)}) + \cdots + y^{(i)}_K\, \log( \hat{p}_K^{(i)}) \big ) \\[.5ex]
\end{align*}
$$

위 식에 사용된 기호의 의미는 다음과 같다.

| 기호 | 의미 |
| :---: | :--- |
| $y^{(i)}_k$ | $i$-번째 입력 샘플이 $k$-번째 범주에 포함되면 1, 아니면 0. |
| $\hat{p}_k^{(i)}$ | $i$-번째 입력 샘플이 $k$-번째 범주에 속할 확률 예측값 |
| $\Theta$ | $\mathbf{\theta}^{(1)}, \dots, \mathbf{\theta}^{(K)}$로 구성된 `(K, n+1)` 모양의 파라미터 행렬 |

**크로스 엔트로피 손실 함수 이해**

틀린 예측을 하면  크로스 엔트로피가 매우 커진다.
예를 들어 하나의 샘플에 대한 손실값은 다음과 같다.

$$
-\big(y_1\, \log ( \hat{p}_1) + \cdots + y_K\, \log( \hat{p}_K)\big)
$$

예를 들어 해당 샘플의 라벨이 $k$라 하자.
그러면 $y_k=1$이고 다른 $y_i$는 모두 0이기에 
손실값은 다음과 같이 매우 단순해진다.

$$
- \log ( \hat{p}_k)
$$

그런데 모델의 예측값이 $k$가 아니라면 $\hat p_k$가 0에 가깝다는 의미이고,
이는 손실값 $-\log ( \hat{p}_k)$ 가 매우 클 수 있음을 의미한다.

:::{admonition} 로그 손실과 크로스 엔트로피
:class: note

$K=2$이면 다음이 성립한다.

$$
y^{(i)}_2 = 1-y^{(i)}_1, \qquad \hat{p}_2^{(i)} = 1- \hat{p}_1^{(i)}
$$

이유는 첫번째 범주에 포함되지 않으면 반드시 두번째 포함됨을 의미하고,
따라서 두번째 범주에 포함될 확률이 바로 1에서 첫번째 범주에 들어갈 확률을 뺀 값이기 때문이다.

결국 $K=2$일 때 크로스 엔트로피 비용함수는 이진 분류 모델인 로지스틱 회귀의 로그 손실 함수와 정확하게 일치한다.

$$
\begin{align*}
J(\Theta) 
& = - \frac{1}{m}\, \sum_{i=1}^{m} \left(y^{(i)}_1\, \log( \hat{p}_1^{(i)}) + (y^{(i)}_2\, \log( \hat{p}_2^{(i)}) \right) \\
& = - \frac{1}{m}\, \sum_{i=1}^{m} \left(y^{(i)}_1\, \log( \hat{p}_1^{(i)}) + (1-y^{(i)}_1)\, \log(1- \hat{p}_1^{(i)}) \right)
\end{align*}
$$
:::

### 붓꽃 다중 클래스 분류

사이킷런의 `LogisticRegression` 예측기를 활용한다.
타깃 라벨이 3개 이상이면 모델이 알아서 다중 클래스 분류기로 훈련한다.
훈련셋과 타깃셋은 다음과 같이 구분되어 있다.

| 구분 | 변수이름 | 설명 |
| :--- | :--- | :--- |
| 훈련셋 | `X` | 꽃잎의 길이와 너비 특성만 포함하는 붓꽃 데이터 샘플<br> 150개로 구성된 넘파이 2차원 어레이 (모양: `(150, 2)`) |
| 타깃 데이터셋 | `y` | 0(세토사), 1(버시컬러), 2(버지니카)로 구성된 넘파이<br> 1차원 어레이 (모양: `(150, )`) |

아래 코드는 꽃잎의 길이와 너비 두 특성을 이용하여 
세토사, 버시컬러, 버지니카 클래스 중 하나를 선택하는 모델을 훈련시킨다.
규제 강도 하이퍼파라미터 `C`를 30으로 지정하여 
L2 규제를 약하게 적용한다.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

softmax_reg = LogisticRegression(l1_ratio=0.0,
                                 C=30,             # 조금 약한 alpha 규제
                                 random_state=42)

softmax_reg.fit(X_train, y_train)
```

학습된 모델의 활용법은 이진 분류기의 경우와 동일하다.
먼저 아래 코드는 테스트셋의 첫 5개 샘플의 품종을 판별한다.

```python
softmax_reg.predict(X_test[:5])
```

결과는 다음과 같이 차례대로 버시컬러, 세토사, 버지니카, 버시컬러, 버시컬러 품종으로 예측되었다.
앞서 이진 분류기에서 예측한 것 처럼 셋째 샘플만 버지니카 품종으로 예측되었다.

```python
array([1, 0, 2, 1, 1])
```

반면에 아래 코드는 테스트셋의 첫 5개 샘플에 대한 범주별 확률을 계산한다.

```python
softmax_reg.predict_proba(X_test[:5])
```

앞서 확인한 것처럼 각 샘플에 대한 예측 품종에 대한 확률이 압도적으로 높다.

```python
array([[5.79887294e-06, 9.82825932e-01, 1.71682688e-02],
       [9.92803024e-01, 7.19697591e-03, 2.46611952e-13],
       [1.85130240e-17, 9.37494824e-07, 9.99999063e-01],
       [1.50163356e-05, 9.39662109e-01, 6.03228752e-02],
       [2.81177214e-06, 8.94089601e-01, 1.05907587e-01]])
```

## 연습문제

**문제 1**

[(코드 워크아웃) 모델 훈련](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-training_models.ipynb)
내용을 학습하라.

**문제 2**

[(Kaggle) Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/)에서 우주선 타이타닉 데이터셋으로 분류 문제를 풀어 보아라.

힌트: [Spaceship Titanic: A complete guide](https://www.kaggle.com/code/samuelcortinhas/spaceship-titanic-a-complete-guide)

**문제 3**

극도로 불균형을 이루는 두 개의 범주로 구성된 신용카드 거래 데이터를 사용하여 사기 탐지 분류 모델을 구축할 때
재현율을 높이는 기법을 확인하라.

참고: [(코드 워크아웃) 신용카드 사기 거래 탐지](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/unbalanced_data_classification.ipynb)

**문제 4**

_주의사항: 이번 문제는 넘파이 어레이를 활용하여 배치 경사하강법의 행렬 연산을 직접 구현한다.따라서 경사하강법 전체 과정과 넘파이 어레이에 대한 깊은 지식을 요구한다._

(1) 붓꽃 데이터셋을 대상으로 다중 클래스 분류를 진행하는 소프트맥스 회귀 모델에
사용되는 배치 경상하강법을 직접 구현하라.
단, 사이킷런은 전혀 사용하지 않고 넘파이 라이브러리만 이용해야 한다.

(2) 가중치들의 노름(norm)의 제곱을 더해서 규제 강도 $\alpha$와 곱한 값을 비용함수에 더하는 규제를 L2 규제(penalty)라 한다.
이전 문제에서 구현한 배치 경사하강법 코드를 수정하여 L2 규제를 지원하는 배치 경상하강법을 구현하라.

(3) 이전 문제의 배치 경사하강법 코드를 수정하여 손실값이 더 이상 개선되지 않으면 학습을 종료시키는 조기 종료 기능을 추가한 배치 경사하강법을 구현하라.


힌트: [소프트맥스 회귀 모델의 배치 경사하강법 적용 훈련 구현](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-training_models.ipynb#scrollTo=5idIMqpwgG2w)